In [ ]:
# runout_probability_with_priority.py
import pandas as pd
import numpy as np
from math import sqrt
from scipy.stats import norm
import os

# ---------- Config ----------
AUG_CSV = "../data stuff/augmented_telemetry_filtered_with_location.csv"
PRED_CSV = "../data stuff/predicted_24h_consumption_per_location.csv"   # must contain 'location_id' and 'model_forecast_l'
Y_CSV = "../data stuff/next24h_consumption.csv"  # optional: contains 'location_id' and 'predicted_demand_l'
OUT_CSV = "../data stuff/runout_probability_per_location_with_priority.csv"

# Assumptions & tuning
ASSUMED_CAN_CAPACITY_L = 20.0
ASSUMED_REFILLS_PER_DAY = 0.2
ALPHA_BLEND = 0.75        # weight analytic probability vs location risk
PRIORITY_THRESHOLD = 0.60 # p_runout > 0.60 -> priority True

# --- Tunable: Supply & Demand (μₛ, σₛ, μ_d, σ_d) and p_analytic ---
# Scale factors (1.0 = use computed values). Applied after per-location computation.
MU_S_SCALE = 1.0          # μₛ — Expected Supply (L/day)
SIGMA_S_SCALE = 1.0       # σₛ — Uncertainty in Supply
MU_D_SCALE = 1.0          # μ_d — Expected Demand (L/day)
SIGMA_D_SCALE = 1.0       # σ_d — Uncertainty in Demand
P_ANALYTIC_SCALE = 1.0    # p_analytic — scale before blending (pure P(runout))

# Optional overrides: if set, same value for all locations (ignores computed).
# Use None to keep computed per-location values.
MU_S_OVERRIDE = None      # μₛ (L/day), or None
SIGMA_S_OVERRIDE = None   # σₛ, or None
MU_D_OVERRIDE = None      # μ_d (L/day), or None
SIGMA_D_OVERRIDE = None   # σ_d, or None

# constants for pressure->liters conversion (if used)
RHO_WATER = 1000.0
G = 9.81
CAN_AREA_M2 = 0.04
PRESSURE_TO_L_FACTOR = (CAN_AREA_M2 / (RHO_WATER * G)) * 1000.0

# ----------------------------

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def estimate_refill_stats(df_aug):
    df = df_aug.copy()
    df = df.sort_values(["device_id", "recorded_at"])
    df["pressure_prev"] = df.groupby("device_id")["pressure_pa"].shift(1)
    refill_mask = df["event_type"] == "REFILL"
    refills = df[refill_mask].dropna(subset=["pressure_prev", "pressure_pa"])
    if len(refills) > 0:
        refills["refill_l"] = (refills["pressure_pa"] - refills["pressure_prev"]) * PRESSURE_TO_L_FACTOR
        refills = refills[refills["refill_l"] > 0]
        refills["date"] = pd.to_datetime(refills["recorded_at"]).dt.date
        grouped = refills.groupby("location_id").agg(
            mean_refill_l=("refill_l", "mean"),
            refills_total=("refill_l", "count"),
            first_date=("date", "min"),
            last_date=("date", "max")
        ).reset_index()
        grouped["days_span"] = (pd.to_datetime(grouped["last_date"]) - pd.to_datetime(grouped["first_date"])).dt.days.clip(lower=1)
        grouped["refills_per_day"] = grouped["refills_total"] / grouped["days_span"]
        return grouped[["location_id", "mean_refill_l", "refills_total", "refills_per_day"]]
    else:
        # empty DataFrame with expected columns
        return pd.DataFrame(columns=["location_id", "mean_refill_l", "refills_total", "refills_per_day"])

def compute_residual_sigma(pred_df, y_df=None):
    """
    Returns DataFrame with columns: location_id, model_forecast_l, sigma_d
    'model_forecast_l' must be present in pred_df.
    If y_df (actuals) is provided, compute residual std per location; else fallback to 30% rule.
    """
    pred = pred_df[["location_id", "model_forecast_l"]].copy()
    if y_df is None or "predicted_demand_l" not in y_df.columns:
        pred["sigma_d"] = 0.30 * pred["model_forecast_l"].replace(0, 1.0)
        return pred

    merged = y_df.merge(pred, on="location_id", how="left").dropna(subset=["model_forecast_l", "predicted_demand_l"])
    if merged.empty:
        pred["sigma_d"] = 0.30 * pred["model_forecast_l"].replace(0, 1.0)
        return pred

    merged["resid"] = merged["predicted_demand_l"] - merged["model_forecast_l"]
    sigma_by_loc = merged.groupby("location_id")["resid"].agg(lambda x: np.std(x, ddof=1) if len(x) > 1 else np.nan).reset_index(name="sigma_d")
    global_sigma = merged["resid"].std(ddof=1)
    sigma_by_loc["sigma_d"] = sigma_by_loc["sigma_d"].fillna(global_sigma if not np.isnan(global_sigma) else 0.30 * merged["predicted_demand_l"].mean())
    mu = merged.groupby("location_id")["model_forecast_l"].mean().reset_index(name="model_forecast_l")
    out = mu.merge(sigma_by_loc, on="location_id", how="left")
    out["sigma_d"] = out["sigma_d"].fillna(0.30 * out["model_forecast_l"].replace(0, 1.0))
    return out

def location_risk_score(centroids_df, depots=None):
    # simple remoteness-based score: 0..1
    if depots is None:
        # fallback: empty -> zero risk
        centroids_df["dist_to_depot_m"] = 0.0
        centroids_df["location_risk"] = 0.0
        return centroids_df[["location_id", "location_risk", "dist_to_depot_m"]]

    DEPOTS = list(depots.values())
    MAX_DIST_FOR_RISK = 200000.0
    min_dists = []
    for la, lo in zip(centroids_df["lat"].astype(float), centroids_df["lon"].astype(float)):
        dlist = [haversine_m(la, lo, dd[0], dd[1]) for dd in DEPOTS]
        min_dists.append(min(dlist))
    centroids_df["dist_to_depot_m"] = min_dists
    centroids_df["remoteness"] = (centroids_df["dist_to_depot_m"] / MAX_DIST_FOR_RISK).clip(0,1)
    centroids_df["location_risk"] = centroids_df["remoteness"]
    return centroids_df[["location_id", "location_risk", "dist_to_depot_m"]]

def run():
    if not os.path.exists(AUG_CSV):
        raise FileNotFoundError(f"Missing file: {AUG_CSV}")
    if not os.path.exists(PRED_CSV):
        raise FileNotFoundError(f"Missing file: {PRED_CSV}")

    df_aug = pd.read_csv(AUG_CSV)
    df_pred = pd.read_csv(PRED_CSV)

    # Ensure required column exists
    if "model_forecast_l" not in df_pred.columns:
        raise KeyError("Predictions file must contain 'model_forecast_l' column.")

    # Estimate refill stats (to derive mu_s and sigma_s)
    refill_stats = estimate_refill_stats(df_aug)

    # build location centroids
    centroids = df_aug.groupby("location_id").agg(
        lat=("lat", "mean"),
        lon=("lon", "mean"),
        records=("location_id", "count")
    ).reset_index()

    # merge refill stats into centroids
    centroids = centroids.merge(refill_stats, on="location_id", how="left")

    # compute mu_s (expected supply in L/day) and sigma_s (supply uncertainty)
    centroids["mu_s"] = np.where(
        centroids["mean_refill_l"].notna(),
        centroids["mean_refill_l"].fillna(ASSUMED_CAN_CAPACITY_L) * centroids["refills_per_day"].fillna(ASSUMED_REFILLS_PER_DAY),
        ASSUMED_CAN_CAPACITY_L * ASSUMED_REFILLS_PER_DAY
    )
    centroids["sigma_s"] = np.where(
        centroids["mean_refill_l"].notna(),
        0.4 * centroids["mu_s"],
        0.8 * centroids["mu_s"]
    )

    # Compute forecast uncertainty sigma_d using available actuals if present
    y_df = pd.read_csv(Y_CSV) if os.path.exists(Y_CSV) else None
    sigma_df = compute_residual_sigma(df_pred, y_df=y_df)

    # merge model forecast and sigma_d into centroids
    # If df_pred has one row per location with model_forecast_l, use it directly; else aggregate
    preds_agg = df_pred.groupby("location_id").agg(model_forecast_l=("model_forecast_l", "mean")).reset_index()
    centroids = centroids.merge(preds_agg, on="location_id", how="left")
    centroids = centroids.merge(sigma_df[["location_id", "sigma_d"]], on="location_id", how="left")

    # fallback sigma_d and model_forecast_l if missing
    centroids["model_forecast_l"] = centroids["model_forecast_l"].fillna(centroids["mu_s"])
    centroids["sigma_d"] = centroids["sigma_d"].fillna(0.30 * centroids["model_forecast_l"].replace(0,1.0))

    # Apply overrides/scales for μₛ, σₛ, μ_d, σ_d
    if MU_S_OVERRIDE is not None:
        centroids["mu_s"] = MU_S_OVERRIDE
    else:
        centroids["mu_s"] = centroids["mu_s"] * MU_S_SCALE
    if SIGMA_S_OVERRIDE is not None:
        centroids["sigma_s"] = SIGMA_S_OVERRIDE
    else:
        centroids["sigma_s"] = centroids["sigma_s"] * SIGMA_S_SCALE
    if MU_D_OVERRIDE is not None:
        centroids["model_forecast_l"] = MU_D_OVERRIDE
    else:
        centroids["model_forecast_l"] = centroids["model_forecast_l"] * MU_D_SCALE
    if SIGMA_D_OVERRIDE is not None:
        centroids["sigma_d"] = SIGMA_D_OVERRIDE
    else:
        centroids["sigma_d"] = centroids["sigma_d"] * SIGMA_D_SCALE

    # compute location risk (remoteness). You can pass real depot coords if available.
    DEPOTS = {
        "Khartoum": (15.5007, 32.5599),
        "PortSudan": (19.6117, 37.2164),
        "ElObeid": (13.1833, 30.2167),
        "Nyala": (12.0667, 24.8667)
    }
    risk_df = location_risk_score(centroids[["location_id","lat","lon"]], depots=DEPOTS)
    centroids = centroids.merge(risk_df, on="location_id", how="left")
    centroids["location_risk"] = centroids["location_risk"].fillna(0.0)

    # Compute analytic probability: p_analytic = P(D > S)
    # Using columns:
    #   model_forecast_l  -> μ_d
    #   sigma_d           -> σ_d
    #   mu_s              -> μ_s
    #   sigma_s           -> σ_s
    mu_d = centroids["model_forecast_l"].astype(float)
    sigma_d = centroids["sigma_d"].astype(float)
    mu_s = centroids["mu_s"].astype(float)
    sigma_s = centroids["sigma_s"].astype(float)

    denom = np.sqrt(np.clip(sigma_d**2 + sigma_s**2, 1e-6, None))
    z = (mu_s - mu_d) / denom
    centroids["p_analytic"] = 1.0 - norm.cdf(z)   # P(D > S)

    # blend with location risk to produce final p_runout
    centroids["p_runout"] = (ALPHA_BLEND * centroids["p_analytic"]) + ((1 - ALPHA_BLEND) * centroids["location_risk"])

    # boolean priority flag when p_runout > PRIORITY_THRESHOLD
    centroids["priority"] = centroids["p_runout"] > PRIORITY_THRESHOLD

    # Save result
    out_cols = [
        "location_id", "lat", "lon", "model_forecast_l", "sigma_d", "mu_s", "sigma_s",
        "p_analytic", "location_risk", "p_runout", "priority"
    ]
    centroids[out_cols].to_csv(OUT_CSV, index=False)
    print(f"Saved runout probabilities with priority flag to: {OUT_CSV}")
    print(centroids[out_cols].head(10))

if __name__ == "__main__":
    run()


Saved runout probabilities with priority flag to: ../data stuff/runout_probability_per_location_with_priority.csv
  location_id        lat        lon  model_forecast_l     sigma_d  \
0     loc_001  15.496613  32.556411       2720.781651  246.599539   
1     loc_002  15.613297  32.530031       2688.766483  246.599539   
2     loc_003  14.729132  33.563615       6607.854027  246.599539   
3     loc_004  13.174927  30.215565       3048.104465  246.599539   
4     loc_005  12.867958  32.951956       3594.513476  246.599539   
5     loc_006  14.059372  30.977965       3727.456391  246.599539   
6     loc_007  15.019627  35.028878       3074.363751  246.599539   
7     loc_008  13.509373  34.014383       3022.776147  246.599539   
8     loc_009  12.505024  30.509872       3000.613578  246.599539   
9     loc_010  15.784521  33.217106       2775.295494  246.599539   

          mu_s      sigma_s  p_analytic  location_risk  p_runout  priority  
0  2348.537547   939.415019    0.649239       0.0